# Train an IGNODE-compatible detector with YOLOX (Ultralytics migration)

**Audience:** customers who started with Ultralytics YOLOv5/v8 and want to migrate to the YOLOX recipe IGNODE uses in production.

**Output:** an ONNX model + sidecar JSON files ready for **Custom Model Upload** in your IGNODE workspace. No retraining inside IGNODE required.

This notebook mirrors the recipe in `ignode-trainer/benchmarks/yolox_upstream_baseline/train_any.py` — the same code IR-3.O Path B production uses.

**Why YOLOX over Ultralytics?**
1. License — Ultralytics requires AGPL-3.0 for derivative works; YOLOX is Apache 2.0.
2. ONNX shape — YOLOX `decode_in_inference=False` matches IGNODE UINF's `yolox_raw` decoder byte-for-byte.
3. Channel order — Megvii uses cv2.imread (BGR). Sidecar declares `BGR`. IGNODE UINF flips channels automatically. No mismatch.

## Steps
1. Mount Google Drive / upload dataset
2. Auto-normalize ANY format (VOC, COCO, YOLOv5/v8, Roboflow exports) → YOLOX's required VOC layout
3. Train
4. Export to ONNX with `decode_in_inference=False`
5. Write the 3 sidecar files (`preprocess_config.json`, `class_labels.json`, optional `manifest.json`)
6. Upload to IGNODE via Custom Model Upload

In [ ]:
# Step 0 — install YOLOX + dependencies. Cell ~3 min on a fresh runtime.
!pip install -q git+https://github.com/Megvii-BaseDetection/YOLOX@0.3.0 --no-deps
!pip install -q supervision==0.21.0 onnx==1.21.0 onnxruntime==1.23.2 'pycocotools>=2.0.8' loguru tabulate ninja
!apt-get install -y -q g++ python3-dev

In [ ]:
# Step 1 — point at your dataset.
# Supported formats (auto-detected): voc, coco, yolov5/v8, roboflow variants.
# Example: drop a Roboflow export ZIP in /content/dataset.zip.

from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/my-detection-dataset'
WORKDIR     = '/content/yolox-run'
!mkdir -p {WORKDIR}
!ls -la {DATASET_DIR}

In [ ]:
# Step 2 — download train_any.py from the benchmark. This script auto-detects
# your dataset format and normalizes it to YOLOX's VOC layout. Same code
# IR-3.O Path B production uses internally.
BENCHMARK_RAW = 'https://bitbucket.org/letscodewithfrancis/ignode-trainer/raw/main/benchmarks/yolox_upstream_baseline'
!curl -fsSL {BENCHMARK_RAW}/train_any.py -o {WORKDIR}/train_any.py
!curl -fsSL {BENCHMARK_RAW}/prepare_dataset.py -o {WORKDIR}/prepare_dataset.py
!curl -fsSL {BENCHMARK_RAW}/voc_eval_patch.py -o {WORKDIR}/voc_eval_patch.py
!ls -la {WORKDIR}

In [ ]:
# Step 3 — train. Defaults match the IR-3.O production recipe:
#   --epochs 300, --batch-size 16, --backbone yolox_s, --fp16, --image-size 640
# For a quick smoke run, drop --epochs to 30.
import os
os.chdir(WORKDIR)
!python train_any.py \
    --data {DATASET_DIR} \
    --output {WORKDIR}/run_001 \
    --epochs 300 \
    --batch-size 16 \
    --backbone yolox_s \
    --image-size 640 \
    --fp16

In [ ]:
# Step 4 — export to ONNX matching IGNODE's UINF yolox_raw contract.
# Critical flag: --decode_in_inference is OMITTED. UINF host-side decodes via
# anchor-grid + stride projection, which requires raw network outputs at
# strides 8/16/32. IR-3.Q catalogs this contract.
import os, shutil
CKPT = f'{WORKDIR}/run_001/yolox_voc_s/best_ckpt.pth'
OUT  = f'{WORKDIR}/run_001/model.onnx'
os.chdir('/usr/local/lib/python3.10/dist-packages/yolox')  # adjust path if torch venv differs
!python tools/export_onnx.py \
    -f /content/yolox-run/exps_yolox_voc_s.py \
    -c {CKPT} \
    --output-name {OUT} \
    --opset 18

In [ ]:
# Step 5 — write the 3 sidecar files IGNODE's Custom Model Upload needs.
# Values MUST match the production trainer's sidecar shape (see IR-3.BB +
# IR-3.CC). The 4 fields that matter most:
#   channel_order: 'BGR'    — Megvii uses cv2.imread
#   rescale:       'none'   — YOLOX trains on raw [0, 255]
#   mean / std:    identity — no normalization
#   postprocess.family: 'yolox_raw' — UINF host-side decode
import json, pathlib

# UPDATE this with YOUR class names, in the SAME ORDER your dataset's
# classes.txt has them. Order is critical — UINF maps model output index N
# to class_labels[N]. Mislabeled = wrong predictions (IR-3.Y trap).
CLASS_LABELS = ['class_0', 'class_1', 'class_2']

preprocess_config = {
    'input_size':      [640, 640],
    'mean':            [0.0, 0.0, 0.0],
    'std':             [1.0, 1.0, 1.0],
    'channel_order':   'BGR',
    'image_format':    'CHW',
    'rescale':         'none',
    'resize_method':   'letterbox',
    'letterbox_color': [114, 114, 114],
    'postprocess': {
        'family':               'yolox_raw',
        'nms_required':         True,
        'confidence_threshold': 0.25,
        'nms_iou_threshold':    0.65,
    },
    '_backbone': 'yolox_s',
}

out = pathlib.Path(f'{WORKDIR}/run_001/upload-bundle')
out.mkdir(exist_ok=True)
shutil.copy(f'{WORKDIR}/run_001/model.onnx', out / 'model.onnx')
(out / 'preprocess_config.json').write_text(json.dumps(preprocess_config, indent=2))
(out / 'class_labels.json').write_text(json.dumps(CLASS_LABELS, indent=2))

print('Upload bundle ready at:', out)
!ls -la {out}

## Step 6 — upload to IGNODE

In your IGNODE workspace:
1. ML Factory → Models → **Upload custom model**
2. Drag the 3 files from `upload-bundle/` into the modal
3. Deploy to a UINF instance
4. Verify in Playground — boxes should land tight on your test image

If the boxes are mispositioned: usually a channel-order or class-label-order issue. The benchmark code on Vultr at `/root/applications/ignode-trainer-git/benchmarks/yolox_upstream_baseline` is the reference recipe; any deviation in the sidecar JSON is the place to debug.